# Guia Completo de Mapas Interativos com Folium
### Referência Prática Organizada em Três Níveis de Complexidade

---
> **Aviso sobre internet:** as parcelas de fundo (tiles) são baixadas de servidores externos. Sem internet o mapa aparece cinza, mas **marcadores, popups e legendas continuam funcionando**.

## Sumário

| Módulo | Nível | Temas |
|--------|-------|-------|
| **1** | Básico | Mapas, Marcadores, Popups e Ícones |
| **2** | Intermediário | FitBounds, Camadas, Grupos e Controles |
| **3** | Avançado | Clusters, Heatmap, Choropleth e Plugins |

---

## Configuração do Ambiente

Execute a célula abaixo: importa o folium e seus plugins, carrega o `cidades.csv` (cidades brasileiras com coordenadas e dados comerciais).

A instalação é feita uma única vez no terminal: `pip install folium`.

In [ ]:
import folium
import numpy as np
import pandas as pd

from folium import plugins
from folium.plugins import HeatMap, MarkerCluster, MiniMap

# Cidades brasileiras com coordenadas e dados de vendas
df = pd.read_csv('cidades.csv', sep=';', encoding='utf-8')

print('Cidades carregadas:', df.shape[0])
df.head()

# MÓDULO 1 — Nível Básico
## Mapas, Marcadores, Popups e Ícones

---
## 1.1 Primeiro Mapa

`folium.Map()` cria um mapa centrado em `location` (latitude, longitude), com zoom inicial dado por `zoom_start`.

**Sintaxe:** `folium.Map(location=[lat, lon], zoom_start=4)`

No notebook, o mapa aparece sozinho ao exibir a variável ao final da célula.

In [ ]:
# Brasil centralizado aproximadamente
m = folium.Map(location=[-15.6, -49.0], zoom_start=4)

m

### Parâmetros importantes

| Parâmetro | Exemplo | Efeito |
|-----------|---------|--------|
| `location` | `[-15.6, -49.0]` | Centro do mapa (latitude, longitude) |
| `zoom_start` | `4` | Nível de zoom inicial (0 = mundo, 18 = rua) |
| `tiles` | `'CartoDB positron'` | Estilo do fundo do mapa |
| `control_scale` | `True` | Mostra escala em km |
| `width` / `height` | `'100%'` / `'600px'` | Tamanho do mapa |

---
## 1.2 Tipos de Fundo (tiles)

O nome em `tiles` troca o estilo do fundo. Os mais usados: `OpenStreetMap`, `CartoDB positron` (claro), `cartodb dark_matter` (escuro), `Esri World Imagery` (satélite) e `OpenTopoMap` (relevo). Também é possível passar uma URL de servidor XYZ.

In [ ]:
m = folium.Map(location=[-15.6, -49.0], zoom_start=4,
               tiles='CartoDB positron', control_scale=True,
               width='100%', height='500px')

m

---
## 1.3 Marcadores

`folium.Marker()` adiciona um pino em uma coordenada. Use `popup` (contento ao clicar) e `tooltip` (ao passar o mouse).

**Sintaxe:** `folium.Marker(location=[lat, lon], popup='Texto', tooltip='Texto')`

In [ ]:
m = folium.Map(location=[-15.6, -49.0], zoom_start=4)

folium.Marker(
    location=[-23.5505, -46.6333],
    popup='São Paulo — maior cidade do país',
    tooltip='Clique em mim'
).add_to(m)

m

---
## 1.4 Ícones, Cores e Popups com HTML

`folium.Icon()` muda a cor e o símbolo do pino (usa a fonte FontAwesome). O `popup` aceita **HTML**, permitindo textos formatados.

**Sintaxe:** `folium.Icon(color='red', icon='info-sign')`

In [ ]:
m = folium.Map(location=[-15.6, -49.0], zoom_start=4)

folium.Marker(
    location=[-23.5505, -46.6333],
    popup='<b>São Paulo</b><br>Região Sudeste',
    tooltip='SP',
    icon=folium.Icon(color='red', icon='info-sign')
).add_to(m)

folium.Marker(
    location=[-3.1190, -60.0217],
    popup='<b>Manaus</b><br>Região Norte',
    tooltip='AM',
    icon=folium.Icon(color='darkblue', icon='cloud')
).add_to(m)

m

---
## 1.5 Círculos: `Circle` e `CircleMarker`

- `folium.Circle`: círculo **com raio em metros** (redimensiona com o zoom).
- `folium.CircleMarker`: círculo **com raio em pixels** (tem tamanho fixo na tela).

**Sintaxe:** `folium.CircleMarker(location=[lat, lon], radius=n, color='#hex', fill=True)`

In [ ]:
m = folium.Map(location=[-15.6, -49.0], zoom_start=4)

folium.Circle(
    location=[-23.5505, -46.6333],
    radius=90000,
    color='crimson',
    fill=True,
    fill_color='crimson',
    popup='Círculo de 90 km em SP'
).add_to(m)

folium.CircleMarker(
    location=[-3.1190, -60.0217],
    radius=12,
    color='#3186cc',
    fill=True,
    fill_color='#3186cc',
    popup='CircleMarker em Manaus'
).add_to(m)

m

---
## 1.6 Muitos Marcadores: Laço sobre os Dados

Com `iterrows()`, percorremos todas as cidades do DataFrame criando um marcador para cada uma. O `icon='tint'` usa um símbolo de gota.

In [ ]:
m = folium.Map(location=[-15.6, -49.0], zoom_start=4)

for _, c in df.iterrows():
    folium.Marker(
        location=[c['lat'], c['lon']],
        popup=f"{c['cidade']} ({c['uf']})",
        tooltip=c['cidade'],
        icon=folium.Icon(color='blue', icon='tint')
    ).add_to(m)

m

---
## 1.7 Salvando o Mapa em HTML

`m.save('arquivo.html')` gera um arquivo HTML **independente**, que pode ser aberto em qualquer navegador ou compartilhado.

In [ ]:
m = folium.Map(location=[-15.6, -49.0], zoom_start=4)

for _, c in df.head(5).iterrows():
    folium.Marker(
        location=[c['lat'], c['lon']],
        popup=c['cidade']
    ).add_to(m)

m.save('mapa_brasil.html')
print('Arquivo mapa_brasil.html gerado!')

m

# MÓDULO 2 — Nível Intermediário
## FitBounds, Camadas, Grupos e Controles

---
## 2.1 Ajuste Automático do Zoom: `fit_bounds`

`fit_bounds()` calcula o **menor retângulo** que contém todos os pontos e ajusta o zoom automaticamente — ideal quando o número de pontos é grande ou desconhecido.

**Sintaxe:** `m.fit_bounds(lista_de_coordenadas)`

In [ ]:
m = folium.Map(location=[-15.6, -49.0], zoom_start=4)

for _, c in df.iterrows():
    folium.CircleMarker(
        location=[c['lat'], c['lon']],
        radius=4,
        color='teal',
        fill=True,
        fill_opacity=0.6,
        tooltip=c['cidade']
    ).add_to(m)

m.fit_bounds([[c['lat'], c['lon']] for _, c in df.iterrows()])

m

---
## 2.2 Múltiplos Fundos com `TileLayer` + `LayerControl`

É possível deixar **vários estilos de fundo disponíveis** para o usuário alternar. O `LayerControl()` cria o seletor de camadas.

**Sintaxe:** `folium.TileLayer('...', name='Nome').add_to(m)`

In [ ]:
m = folium.Map(location=[-15.6, -49.0], zoom_start=4, control_scale=True)

folium.TileLayer('cartodb dark_matter', name='Escuro noturno').add_to(m)
folium.TileLayer(tiles='OpenTopoMap', name='Relevo').add_to(m)
folium.TileLayer(tiles='Esri World Imagery', name='Satélite').add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m

---
## 2.3 Agrupando Marcadores em `FeatureGroup`

`FeatureGroup` junta marcadores em uma **camada ligável/desligável**. Combinado com `LayerControl`, o usuário escolhe quais grupos ver.

**Sintaxe:** `folium.FeatureGroup(name='Norte')`

In [ ]:
m = folium.Map(location=[-15.6, -49.0], zoom_start=4)

grupos = {
    'Norte': folium.FeatureGroup(name='Região Norte'),
    'Sul': folium.FeatureGroup(name='Região Sul'),
    'Demais': folium.FeatureGroup(name='Demais regiões')
}

for _, c in df.iterrows():
    grupo = 'Norte' if c['regiao'] == 'Norte' else ('Sul' if c['regiao'] == 'Sul' else 'Demais')
    folium.CircleMarker(
        location=[c['lat'], c['lon']],
        radius=6,
        color='darkgreen',
        fill=True,
        fill_color='limegreen',
        popup=c['cidade'],
        tooltip=c['cidade']
    ).add_to(grupos[grupo])

for grupo in grupos.values():
    grupo.add_to(m)

folium.LayerControl().add_to(m)

m

---
## 2.4 Popups Avançados com HTML e Dados do DataFrame

F-string com HTML gera um **painel informativo** no popup, usando os Dados de cada cidade. `folium.Popup(max_width=...)` limita a largura.

In [ ]:
m = folium.Map(location=[-15.6, -49.0], zoom_start=4)

for _, c in df.head(6).iterrows():
    html = f'''
        <b>{c['cidade']} — {c['uf']}</b><br>
        Região: {c['regiao']}<br>
        População: {c['populacao_milhoes']:.1f} mi<br>
        Vendas anuais: R$ {c['vendas_anuais']:.0f} milhões
    '''
    folium.Marker(
        location=[c['lat'], c['lon']],
        popup=folium.Popup(html, max_width=250),
        tooltip=c['cidade']
    ).add_to(m)

m

---
## 2.5 Ferramentas Auxiliares: Medição e Localização

Os plugins `MeasureControl` (medir distâncias) e `LocateControl` (localizar o usuário) adicionam **botões funcionais** ao mapa.

In [ ]:
m = folium.Map(location=[-15.6, -49.0], zoom_start=4)

plugins.MeasureControl(
    position='topright',
    primary_length_unit='kilometers'
).add_to(m)

plugins.LocateControl(position='bottomright').add_to(m)

m

# MÓDULO 3 — Nível Avançado
## Clusters, Heatmap, Choropleth e Plugins

---
## 3.1 `MarkerCluster` — Agrupamento Automático de Marcadores

Com centenas de pontos, o `MarkerCluster` **agrupa marcadores próximos em balões numerados**, que se expandem ao aumentar o zoom.

**Sintaxe:** `cluster = MarkerCluster().add_to(m)` — depois, cada marcador é `add_to(cluster)`

In [ ]:
# Para demonstrar o cluster, duplicamos cada cidade em 3 dimensões próximas
rng = np.random.default_rng(42)
pontos = []
for _, c in df.iterrows():
    for _ in range(3):
        pontos.append([
            c['lat'] + rng.uniform(-0.15, 0.15),
            c['lon'] + rng.uniform(-0.15, 0.15),
            c['cidade']
        ])

m = folium.Map(location=[-15.6, -49.0], zoom_start=4)
cluster = MarkerCluster(name='Cidades').add_to(m)

for lat, lon, cidade in pontos:
    folium.Marker(
        location=[lat, lon],
        popup=cidade,
        icon=folium.Icon(color='green')
    ).add_to(cluster)

folium.LayerControl().add_to(m)

m

---
## 3.2 `HeatMap` — Mapa de Calor

O plugin `HeatMap` recebe uma lista de `[lat, lon, intensidade]` e gera um **gradiente de densidade**. Ótimo para visualizar concentração de ocorrências.

**Sintaxe:** `HeatMap(lista_de_pontos, radius=25).add_to(m)`

In [ ]:
m = folium.Map(location=[-15.6, -49.0], zoom_start=4, tiles='CartoDB positron')

dados_calor = df[['lat', 'lon', 'vendas_anuais']].values.tolist()
HeatMap(dados_calor, radius=25, blur=15, max_zoom=1).add_to(m)

m

---
## 3.3 `MiniMap` e `Fullscreen` — Navegação

O `MiniMap` mostra um mapa em miniatura no canto (com botão para abrir/fechar) e o `Fullscreen` adiciona o botão de tela cheia.

In [ ]:
m = folium.Map(location=[-15.6, -49.0], zoom_start=4, control_scale=True)

for _, c in df.head(8).iterrows():
    folium.CircleMarker(
        location=[c['lat'], c['lon']],
        radius=8,
        color='green',
        fill=True,
        fill_color='green'
    ).add_to(m)

MiniMap(toggle_display=True, zoom_level_offset=-5).add_to(m)
plugins.Fullscreen(position='topright').add_to(m)

m

---
## 3.4 `Choropleth` — Mapa Temático por Região

O `Choropleth` **pinta polígonos** conforme um valor associado. Os polígonos vêm de um GeoJSON; abaixo usamos **polígonos aproximados** das 5 regiões do Brasil e colorimos pelo total de vendas.

**Sintaxe:** `folium.Choropleth(geo_data=geojson, data=df, columns=['chave', 'valor'], key_on='feature.properties.chave', fill_color='YlGnBu')`

> Em projetos reais, use o GeoJSON oficial do IBGE de estados ou municípios.

In [ ]:
# Polígonos aproximados das 5 regiões (esquema [longitude, latitude])
regioes_geo = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "properties": {"regiao": "Centro-Oeste"},
         "geometry": {"type": "Polygon", "coordinates": [[[-62, -8], [-47, -8], [-47, -16], [-54, -18], [-62, -23]]]}},
        {"type": "Feature", "properties": {"regiao": "Nordeste"},
         "geometry": {"type": "Polygon", "coordinates": [[[-47, -1], [-35, -1], [-35, -8], [-39, -11], [-46, -16]]]}},
        {"type": "Feature", "properties": {"regiao": "Norte"},
         "geometry": {"type": "Polygon", "coordinates": [[[-74, 5], [-44, 5], [-44, -2], [-51, -10], [-74, -8]]]}},
        {"type": "Feature", "properties": {"regiao": "Sudeste"},
         "geometry": {"type": "Polygon", "coordinates": [[[-53, -15], [-41, -15], [-41, -25], [-53, -25]]]}},
        {"type": "Feature", "properties": {"regiao": "Sul"},
         "geometry": {"type": "Polygon", "coordinates": [[[-57, -23], [-48, -23], [-48, -33.7], [-57, -33.7]]]}}
    ]
}

vendas_regioes = df.groupby('regiao', as_index=False)['vendas_anuais'].sum()

m = folium.Map(location=[-15.6, -49.0], zoom_start=4, tiles='CartoDB positron')

folium.Choropleth(
    geo_data=regioes_geo,
    name='Vendas por Região',
    data=vendas_regioes,
    columns=['regiao', 'vendas_anuais'],
    key_on='feature.properties.regiao',
    fill_color='YlGnBu',
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name='Vendas anuais (R$ milhões)'
).add_to(m)

folium.LayerControl().add_to(m)

m

---
## 3.5 Projeto Integrador: O Mapa Final

Combinando coroplet, cluster de cidades, controle de camadas e tela cheia em **um único mapa** exportado para HTML.

In [ ]:
cores = {
    'Norte': 'green',
    'Nordeste': 'orange',
    'Centro-Oeste': 'purple',
    'Sudeste': 'firebrick',
    'Sul': 'darkblue'
}

m = folium.Map(location=[-15.6, -49.0], zoom_start=4,
               tiles='CartoDB positron', control_scale=True)

# Camada 1: coroplet das regiões
folium.Choropleth(
    geo_data=regioes_geo,
    name='Vendas por Região',
    data=vendas_regioes,
    columns=['regiao', 'vendas_anuais'],
    key_on='feature.properties.regiao',
    fill_color='YlGnBu',
    fill_opacity=0.55,
    line_opacity=0.3,
    legend_name='Vendas anuais (R$ milhões)'
).add_to(m)

# Camada 2: cluster de cidades coloridas por região
cluster = MarkerCluster(name='Cidades').add_to(m)
for _, c in df.iterrows():
    folium.CircleMarker(
        location=[c['lat'], c['lon']],
        radius=6,
        color=cores[c['regiao']],
        fill=True,
        fill_color=cores[c['regiao']],
        fill_opacity=0.85,
        popup=f"<b>{c['cidade']}</b><br>Região: {c['regiao']}<br>Vendas: R$ {c['vendas_anuais']:.0f} mi",
        tooltip=c['cidade']
    ).add_to(cluster)

folium.LayerControl(collapsed=False).add_to(m)
plugins.Fullscreen(position='topright').add_to(m)

m.save('mapa_final.html')
print('Arquivo mapa_final.html gerado!')

m

---
# Resumo Rápido de Referência

| Função | Descrição |
|--------|-----------|
| `folium.Map()` | Cria o mapa (location, zoom_start, tiles) |
| `folium.Marker()` | Pino com popup/tooltip/ícone |
| `folium.Circle()` | Círculo com raio em metros |
| `folium.CircleMarker()` | Círculo com raio em pixels |
| `folium.Icon()` | Cor e símbolo do marcador |
| `folium.Popup()` | Popup (aceita HTML e max_width) |
| `folium.TileLayer()` | Estilo adicional de fundo |
| `folium.LayerControl()` | Seletor de camadas |
| `folium.FeatureGroup()` | Agrupa objetos em camada própria |
| `folium.Choropleth()` | Mapa temático por polígono (GeoJSON) |
| `m.fit_bounds(...)` | Ajusta o zoom para cobrir os pontos |
| `m.save('mapa.html')` | Exporta o mapa para HTML |
| `MarkerCluster` | Agrupa marcadores próximos (plugin) |
| `HeatMap` | Mapa de calor por densidade (plugin) |
| `MiniMap` | Mapa em miniatura (plugin) |
| `plugins.Fullscreen` | Botão de tela cheia |
| `plugins.MeasureControl` | Medir distâncias |
| `plugins.LocateControl` | Localizar o usuário |